# Lab 5 — Twitter Sentiment: Mini-Kaggle con Transformers

**Tarea:** clasificar tweets como positivos (`pos`) o negativos (`neg`) usando modelos pre-entrenados de Hugging Face.  
**Tu palanca:** elegir el mejor modelo. No se entrena nada — solo inferencia.

**Métrica:** F1-score (pos) · Leaderboard: URL del profesor

In [ ]:
import os, re, time, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
import pandas as pd
import requests
from pathlib import Path
from sklearn.metrics import f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

SERVER_URL = 'https://nlp-lab5-818627151997.europe-southwest1.run.app'

# ── Datos ─────────────────────────────────────────────────────────────────────
try:
    import google.colab; _DATA = Path('/content/data')
except ImportError:
    _DATA = Path('../labs/lab5/data')
_DATA.mkdir(parents=True, exist_ok=True)

for _f in ('train.csv', 'public_test.csv'):
    if not (_DATA / _f).exists():
        (_DATA / _f).write_bytes(
            requests.get(f'{SERVER_URL}/data/{_f}', timeout=30).content)

train = pd.read_csv(_DATA / 'train.csv')
test  = pd.read_csv(_DATA / 'public_test.csv')

# ── Limpieza de tweets ────────────────────────────────────────────────────────
def clean_tweet(text):
    text = re.sub(r'http\S+|www\S+', '', str(text))
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    return re.sub(r'\s+', ' ', text).strip()

# ── Carga de modelo ───────────────────────────────────────────────────────────
def make_clf(model_name, batch_size=16):
    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()
    label_map = model.config.id2label
    def classify(texts):
        if isinstance(texts, str): texts = [texts]
        out = []
        for i in range(0, len(texts), batch_size):
            b = texts[i:i+batch_size]
            with torch.no_grad():
                ids    = tok(b, return_tensors='pt', truncation=True,
                             padding=True, max_length=128)
                logits = model(**ids).logits
                pred   = logits.argmax(dim=1).tolist()
                score  = torch.softmax(logits, dim=1).max(dim=1).values.tolist()
            out += [{'label': label_map[p], 'score': s} for p, s in zip(pred, score)]
        return out
    classify.label_map = label_map
    return classify

def to_label(hf_label):
    return 'pos' if hf_label.upper() in ('POSITIVE', 'LABEL_2', 'POS') else 'neg'

# ── Evaluación y envío ────────────────────────────────────────────────────────
def evaluar(clf, n=200):
    sample = train.sample(n=n, random_state=42)
    texts  = [clean_tweet(t) for t in sample['text']]
    y_pred = [to_label(r['label']) for r in clf(texts)]
    y_true = sample['label'].tolist()
    print(f'F1 local (pos) sobre {n} tweets: {f1_score(y_true, y_pred, pos_label="pos"):.4f}')
    print(classification_report(y_true, y_pred, target_names=['neg', 'pos']))

def enviar(clf):
    texts   = [clean_tweet(t) for t in test['text']]
    t0      = time.time()
    results = clf(texts)
    print(f'Inferencia: {time.time()-t0:.1f}s')
    payload = [{'id': int(row.id), 'label': to_label(results[i]['label'])}
               for i, row in enumerate(test.itertuples())]
    r = requests.post(f'{SERVER_URL}/submit',
                      json={'team': ALUMNO, 'predictions': payload}, timeout=60)
    if r.status_code == 429:
        print(f'⏳ {r.json()["detail"]}')
    else:
        d = r.json()
        print(f"✅ F1: {d['f1']:.4f}  |  Rank: #{d['rank']}")

print(f'Train: {len(train):,}  |  Test: {len(test):,}  |  '
      f'pos: {train.label.eq("pos").mean():.0%}  '
      f'neg: {train.label.eq("neg").mean():.0%}')

---
## ✏️ Tu nombre

In [2]:
ALUMNO = "pablo"   # <-- escribe tu nombre o alias

---
## 1. ¿Cómo son los tweets?

Echa un vistazo al dataset antes de buscar un modelo.

In [3]:
print(f'Longitud media: {train.text.str.split().str.len().mean():.0f} palabras\n')
for label, emoji in [('pos', '😊'), ('neg', '😞')]:
    for txt in train[train.label == label].text.head(2):
        print(f'{emoji}  {txt[:120]}')
    print()

Longitud media: 19 palabras

😊  RT @user #jeopardy Where everyone will be watching the Ravens vs Eagles game THIS Thursday... What is @user ?  #goodAnsw
😊  "All you HNN All-Stars know who I want to send a big birthday shoutout to...Prince George!  Happy 2nd birthday, your fut

😞  ordered my back pack off of Amazon 3 weeks ago and it turns out they lost my package and school starts tomorrow #sick
😞  @user YOU BROKE THE US TRADE EMBARGO WITH CUBA AND BRAGGED ABOUT IT?!!! What is wrong with you? #trumpthecrook



---
## 2. Baseline

Este modelo fue entrenado en reseñas de cine, no en tweets. Es el punto de referencia a superar.

In [4]:
clf_base = make_clf('distilbert-base-uncased-finetuned-sst-2-english')
print('Etiquetas:', clf_base.label_map)
evaluar(clf_base)

Etiquetas: {0: 'NEGATIVE', 1: 'POSITIVE'}
F1 local (pos) sobre 200 tweets: 0.7801
              precision    recall  f1-score   support

         neg       0.58      0.79      0.67        67
         pos       0.87      0.71      0.78       133

    accuracy                           0.73       200
   macro avg       0.72      0.75      0.72       200
weighted avg       0.77      0.73      0.74       200



---
## 3. Tu modelo

Busca un modelo mejor en [huggingface.co/models](https://huggingface.co/models):

- Filtra por **Text Classification** + **English**
- Busca `twitter sentiment` o `tweet sentiment`
- Lee la model card: ¿en qué datos fue entrenado? ¿qué F1 reporta?
- Elige variantes `base` o `distil` — los `large` son lentos en CPU

Cuando cargues el modelo, fíjate en las etiquetas que imprime. La función `to_label` mapea `POSITIVE` / `LABEL_2` / `POS` → `pos` y el resto → `neg`. Si tu modelo usa otros nombres, ajústala en la celda de setup.

**Modelos de referencia** (punto de partida, no la respuesta):

| Modelo | Entrenado en |
|--------|--------------|
| `distilbert-base-uncased-finetuned-sst-2-english` | Reseñas de cine |
| `cardiffnlp/twitter-roberta-base-sentiment-latest` | Tweets EN |
| `finiteautomata/bertweet-base-sentiment-analysis` | Tweets EN |
| `pysentimiento/robertuito-sentiment-analysis` | Tweets ES/EN |

In [5]:
MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment-latest'  # <-- cambia esto

clf = make_clf(MODEL_NAME)
print('Etiquetas:', clf.label_map)
evaluar(clf)

Etiquetas: {0: 'negative', 1: 'neutral', 2: 'positive'}
F1 local (pos) sobre 200 tweets: 0.8498
              precision    recall  f1-score   support

         neg       0.66      0.99      0.79        67
         pos       0.99      0.74      0.85       133

    accuracy                           0.82       200
   macro avg       0.82      0.86      0.82       200
weighted avg       0.88      0.82      0.83       200



In [6]:
enviar(clf)

Inferencia: 8.7s
✅ F1: 0.8759  |  Rank: #1


---
## Tips

- **El dominio importa más que el tamaño** — un modelo `base` entrenado en tweets suele ganar a un `large` entrenado en reseñas
- **Etiquetas inesperadas** — si el modelo usa nombres no reconocidos por `to_label`, verás que todo predice `neg`; ajusta la función
- **Modelos de 3 clases** (negative / neutral / positive) — los neutrales se mapean a `neg`; puede que cambiar esto mejore el F1
- **El F1 local es orientativo** — 200 tweets tiene varianza; envía cuando veas resultados estables

¡Buena suerte! 🏆